In [1]:
import xgi
import numpy as np
import json
import pickle
from multiprocess import Pool
from tqdm import tqdm
from had_model import *

In [3]:
# upload SocioPatterns hypergraph
tag = 'SFHH'
with open(f'../data/SocioPatterns/aggr_15min_cliques_thr1_{tag}.json') as file: 
            data = json.load(file)
H = xgi.from_hyperedge_list(data)
# relabel and remove eventual multiple edges
H.cleanup(isolates=True, singletons=True, connected=False)
N = len(H.nodes)

# hyperdegrees in the initial hypergraph
counts_hd_0 = dict()
hds_0 = H.nodes.degree.asdict()
for k in sorted(hds_0.values()):
    if k in counts_hd_0.keys():
        counts_hd_0[k]+=1
    else:
        counts_hd_0[k]=1        

print(H, '\nHyperedge sizes:', xgi.unique_edge_sizes(H), '\nConnected:',  xgi.is_connected(H))

Unnamed Hypergraph with 403 nodes and 6398 hyperedges 
Hyperedge sizes: [2, 3, 4, 5, 6, 7, 8, 9, 10] 
Connected: True


# Results as a function of $\epsilon$

In [9]:
iterations = 20     # number of runs for each epsilon
n_process = 7       # number of processes for parallelization

### SET PARAMETERS OF THE MODEL
cond = 'max_min'    # 'max_min' or 'std'
epsilons = np.linspace(0., 0.6, num=25)[1:]
if cond=='std':
    epsilons = np.linspace(0., 0.4, num=17)
epsilons[0] = 0.01   # replace eps=0 with eps=0.01

ncc = []
scc1 = []
scc2 = []
var_scc1 = []
var_scc2 = []
nhe = []
s_avg = []
s_max = []
n_tsteps = []

         
for eps in tqdm(epsilons):

    p = Pool(processes=n_process)
    args = [(H, eps, cond)] * iterations
    res_runs = p.map(get_results_SP, args)

    ncc_it = []
    scc1_it = []
    scc2_it = []
    nhe_it = []
    s_avg_it = []
    s_max_it = []
    n_tsteps_it = []

    # loop over iterations
    for res in res_runs:     
        
        groups, opinions, ng_0 = res[0], res[1], res[2]
        n_tsteps_it.append(len(groups))
        # number of connected components and size of the two largest
        hedges = groups[-1]
        h = xgi.Hypergraph(hedges)
        # list of connected components
        cc = [j for j in xgi.connected_components(h)]
        ncc_it.append( len(cc) )                    
        scc = set([len(i) for i in cc])
        scc1_it.append( max(scc) / N )
        scc.remove(max(scc))
        if scc:
            scc2_it.append( max(scc) / N )
        else:
            scc2_it.append(0)

        # relative number of hyperedges
        nhe_it.append( len(groups[-1]) / ng_0 )
        # relative avg and std of hyperedge sizes
        sizes_end_it = [len(e) for e in groups[-1]]
        s_avg_it.append( np.mean(sizes_end_it) )
        s_max_it.append( np.max(sizes_end_it) )
        
    ncc.append(np.mean(ncc_it))
    scc1.append(np.mean(scc1_it))
    scc2.append(np.mean(scc2_it))
    var_scc1.append(np.var(scc1_it))
    var_scc2.append(np.var(scc2_it))
    nhe.append(np.mean(nhe_it))
    s_avg.append(np.mean(s_avg_it))
    s_max.append(np.mean(s_max_it))
    n_tsteps.append(np.mean(n_tsteps_it))

results = {
    'ncc': ncc,
    'scc1': scc1,
    'scc2': scc2,
    'var_scc1': var_scc1,
    'var_scc2': var_scc2,
    'nhe': nhe,
    's_avg': s_avg,
    's_max': s_max,
    'n_timesteps': n_tsteps,
    'epsilons': epsilons,
    'n_iter': iterations
    }

# save results
with open(f'../results/SP_{tag}_{cond}.pkl', 'wb') as fp:
        pickle.dump(results, fp)

100%|███████████████████████████████████████████| 24/24 [18:02<00:00, 45.12s/it]


# Various distributions for fixed $\epsilon$

In [5]:
iterations = 50     # number of runs for each epsilon
n_process = 7       # number of processes for parallelization

cond = 'max_min'    # 'max_min' or 'std'

for epsilon in tqdm([0.1, 0.3, 0.4]):
    
    counts_cc = dict()
    counts_he = dict()
    counts_hd = dict()
    
    p = Pool(processes=n_process)
    args = [(H, epsilon, cond)] * iterations
    res_runs = p.map(get_results_SP, args)
    
    # loop over iterations
    for res in res_runs:     
        
        groups, opinions, ng_0 = res[0], res[1], res[2]
        # opinions and groups at the end of the simulation
        op_fin = [op_n_t[-1] for op_n_t in opinions.values()]
        hedges = groups[-1]
        h = xgi.Hypergraph(hedges)
        op_fin_cc = [opinions[list(cc)[0]][-1] for cc in xgi.connected_components(h)]
        
        # sizes of connected components
        scc = [len(i) for i in xgi.connected_components(h)]
        for s in sorted(scc):
            if s in counts_cc.keys():
                counts_cc[s]+=1
            else:
                counts_cc[s]=1
    
        # sizes of hyperedges
        she = [len(i) for i in h.edges.members()]
        for s in sorted(she):
            if s in counts_he.keys():
                counts_he[s]+=1
            else:
                counts_he[s]=1
    
        # hyperdegrees
        hds = h.nodes.degree.aslist()
        for deg in sorted(hds):
            if deg in counts_hd.keys():
                counts_hd[deg]+=1
            else:
                counts_hd[deg]=1
        
    sizes_cc = {s: c/iterations for s,c in counts_cc.items()}
    sizes_he = {s: c/iterations for s,c in counts_he.items()}
    hyperdegs = {d: c/iterations for d,c in counts_hd.items()}
    
    results = {
        'epsilon': epsilon,
        'n_iter': iterations,
        'sizes_cc': sizes_cc,
        'sizes_he': sizes_he,
        'hyperdegs': hyperdegs,
        'hyperdegs_0': counts_hd_0,
        'op_fin_1run': op_fin,
        'op_fin_cc_1run': op_fin_cc
        }
    
    # save results
    with open(f'../results/SP_{tag}_{cond}_distrib_eps_{epsilon}.pkl', 'wb') as fp:
            pickle.dump(results, fp)

100%|████████████████████████████████████████████| 3/3 [06:36<00:00, 132.05s/it]
